# 📘 Notebook: Geometry Optimization in a Cavity (CQED-DFT / CQED-HF)

## 🧠 Overview

In this notebook, we perform a geometry optimization of a molecule inside an optical cavity using:

- CQED-SCF (RHF or DFT)
- Analytic gradients
- BFGS optimization

We will:

- Optimize a molecular structure
- Track the optimization trajectory
- Write results to an .xyz file for visualization

In [1]:
import numpy as np
import psi4

#psi4.core.be_quiet()

from cqed_rhf import CQEDRHFCalculator
from cqed_rhf.drivers import bfgs_optimize
from cqed_rhf.utils import write_xyz, ANGSTROM_TO_BOHR

# 🧬 Molecular Geometry

We define the molecule in Angstroms.

⚠️ Important: We enforce no_reorient and no_com so the cavity field remains well-defined.

In [2]:
geometry = """
1 1
C                  0.51932475    1.23303451   -0.03194925
C                  1.94454413    1.26916358   -0.03672882
C                  2.62037793    0.09283428   -0.02499003
C                 -0.19603352    0.03013062    0.00102732
H                 -0.02069420    2.17423764   -0.04336646
H                  2.48281698    2.20891057   -0.03611879
H                 -1.27770137    0.03990295    0.01166953
N                  4.09213475    0.09594076    0.03662979
O                  4.63930696   -1.02169275    0.14459220
O                  4.66489883    1.19839699   -0.02327545
C                  0.49428518   -1.16712649    0.02099746
H                 -0.03251071   -2.11492669    0.05447935
C                  1.96291176   -1.21653219   -0.02111314
H                  2.44359113   -1.96306433    0.61513886
Br                 2.17304025   -1.94912156   -1.90618750
units angstrom
no_reorient
no_com
symmetry c1
"""

# 🖥️ Psi4 Options

In [3]:
psi4_options = {
    "basis": "6-31G",
    "scf_type": "df",
    "e_convergence": 1e-12,
    "d_convergence": 1e-12,
}

psi4.set_options(psi4_options)

# 🔦 Cavity Parameters

We define the cavity coupling:

In [4]:
lambda_vector = [0, 0, 0] #[0.078, 0.055, 0.027]
omega = 0.06615

# 🧮 CQED Calculator

You can switch between:

- RHF (no functional)
- DFT (e.g., "wb97x-d")



In [5]:
calc = CQEDRHFCalculator(
    lambda_vector=lambda_vector,
    psi4_options=psi4_options,
    omega=omega,
    density_fitting=True,
    charge=1,
    multiplicity=1,
    functional="wb97x",  # try None for RHF
)

# 📁 Prepare XYZ Trajectory Output

This allows visualization of the optimization path

In [6]:
xyz_file = "optimization_trajectory.xyz"

# clear existing file
open(xyz_file, "w").close()

mol = psi4.geometry(geometry)
symbols = [mol.symbol(i) for i in range(mol.natom())]

# Run BFGS Optimization
Using analytic gradients and Psi4's DF machinery for J and K gradients

In [7]:
print("Starting geometry optimization...\n")

opt_result = bfgs_optimize(
    calculator=calc,
    geometry=geometry,
    canonical="psi4",   # uses DF gradient machinery
    gtol=1e-6,
    maxiter=50,          # just taking a few steps here to demo, want to increase this!
    debug=True,         # writes trajectory during optimization
)

Starting geometry optimization...

Starting BFGS optimization...
Going to update for 50 iterations or until gradient norm < 1.00e-06 Ha/bohr

Running CQED-SCF energy calculation...

Functional: wb97x
Starting CQED-SCF calculation...
Method: RKS
Functional: wb97x
Using density fitting through Psi4 JK.
Psi4 canonical gradient time: 3.1524 s
Quadrupole gradient time: 0.0048 s
Dipole–dipole gradient time: 0.0320 s
Dispersion correction energy: 0.000000000000 Eh
Total energy (CQED + dispersion): -3009.851691031422 Eh
Dispersion correction gradient norm: 0.000000e+00 Eh/Bohr

Running CQED-SCF energy calculation...

Functional: wb97x
Starting CQED-SCF calculation...
Method: RKS
Functional: wb97x
Using density fitting through Psi4 JK.
Psi4 canonical gradient time: 3.1866 s
Quadrupole gradient time: 0.0050 s
Dipole–dipole gradient time: 0.0199 s
Dispersion correction energy: 0.000000000000 Eh
Total energy (CQED + dispersion): -3009.851691031422 Eh
Dispersion correction gradient norm: 0.000000e+

/Users/jfoley19/miniconda3/envs/psi4_new/lib/python3.11/site-packages/scipy/optimize/_minimize.py:733: OptimizeWarning: Desired error not necessarily achieved due to precision loss.
  res = _minimize_bfgs(fun, x0, args, jac, callback, **options)


# Save final geometry

In [8]:
coords_opt_bohr = opt_result.x.reshape(-1, 3)
coords_opt_angstrom = coords_opt_bohr / ANGSTROM_TO_BOHR

write_xyz(
    xyz_file,
    symbols,
    coords_opt_angstrom,
    comment=f"FINAL OPTIMIZED | E = {opt_result.fun:.10f} Ha",
    mode="a",
)

AttributeError: 'tuple' object has no attribute 'x'

In [9]:
print(opt_result)

(  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: -3009.8522662838377
        x: [ 9.804e-01  2.335e+00 ... -3.804e+00 -3.605e+00]
      nit: 30
      jac: [-1.745e-05 -8.473e-05 ...  7.137e-05 -1.732e-04]
 hess_inv: [[ 1.074e+00 -1.010e-01 ...  9.843e-01 -3.600e-01]
            [-1.010e-01  1.181e+00 ... -1.756e+00  2.780e-01]
            ...
            [ 9.843e-01 -1.756e+00 ...  3.005e+01 -6.476e+00]
            [-3.600e-01  2.780e-01 ... -6.476e+00  4.329e+00]]
     nfev: 69
     njev: 59, {})


# Print Summary

In [ ]:
print("\nOptimization finished.")
print(f"Converged: {opt_result.success}")
print(f"Final energy (Ha): {opt_result.fun:.10f}")
print(f"XYZ trajectory written to: {xyz_file}")